# `Maranatha.jl` :: Practical Reporting Workflow

This notebook demonstrates a realistic end-to-end workflow for
analyzing a nontrivial quadrature problem and producing reproducible
research artifacts using `Maranatha.jl`.

Unlike the earlier pedagogical examples, this notebook uses a
physically motivated and numerically challenging integrand
```math
F_{0000} − \gamma_E + 1 
```
and walks through the complete reporting pipeline
that would typically be used in an actual research project.

The workflow combines multiple stages of a real numerical study:

- running a concrete quadrature setup,
- inspecting raw convergence datapoints,
- generating datapoints-only reports,
- performing continuum extrapolation,
- visualizing fitted results,
- producing publication-ready summaries and internal notes.

The goal is to show how a single experiment can be transformed into a
self-contained documentation package suitable for archival,
collaboration, or inclusion in technical reports.


## Initialize Julia environment

First we configure the Julia runtime and load the
**`Maranatha.jl`** package.

In [ ]:
using Maranatha

## Step 1 :: Define the experiment

We begin by loading a realistic test integrand and defining the
quadrature configuration.

The integrand used here corresponds to the quantity

```math
F_{0000} - \gamma_E + 1,
```

which is known to be numerically demanding due to its structure.
Such examples are useful for stress-testing convergence behavior and
error-estimation strategies.

We then specify:

- the integration domain,
- the quadrature rule,
- the sampling resolutions,
- the error-estimation method,
- output and reporting options.

In [2]:
include("../test/complicated/experiments/integrand_F0000GammaEminus1.jl")

register_F0000_integrand!()

ff(x)  = g_F0000_pure(x)

bounds = (0.0, 1.0)
use_error_jet = true
dim = 1
rule = :gauss_p12
boundary = :LU_EXEX
ns = [3,4,5,6,7,8,9]
ns .+= 14
err_method = :refinement # :forwarddiff , :taylorseries , :enzyme , :fastdifferentiation
nerr_terms = 3
ff_shift = 0
fit_terms = 4
result_string = "F0000"
save_path = joinpath("../samples", "jld2")
write_summary = true
save_file = true
use_cuda = false

false

## Step 2 :: Run the quadrature study

The call to `run_Maranatha` evaluates the integral at multiple
resolutions and constructs a structured dataset containing:

- step sizes,
- quadrature estimates,
- modeled error scales,
- metadata describing the experiment.

If file output is enabled, the dataset is also saved to disk in JLD2
format for later reuse or independent analysis.

In [ ]:
Maranatha.Utils.JobLoggerTools.log_stage_benji("BEGIN run_Maranatha")

run_result = run_Maranatha(
    ff, 
    bounds...; 
    dim=dim, 
    nsamples=ns,
    rule=rule, 
    boundary=boundary, 
    err_method=err_method,
    fit_terms=fit_terms, 
    nerr_terms=nerr_terms,
    ff_shift=ff_shift, 
    use_error_jet=use_error_jet,
    name_prefix=result_string,
    save_path=save_path,
    write_summary=write_summary,
    use_cuda=use_cuda
)

Maranatha.Utils.JobLoggerTools.log_stage_benji("END run_Maranatha")

[2026-03-19 14:14:51.577] 
[2026-03-19 14:14:51.577] ==================================================
[2026-03-19 14:14:51.577] >>> BEGIN run_Maranatha
[2026-03-19 14:14:51.577] ==================================================
[2026-03-19 14:14:51.577] 
[2026-03-19 14:14:55.879] 
[2026-03-19 14:14:55.879] ==================================================
[2026-03-19 14:14:55.879] >>> Start run_Maranatha
[2026-03-19 14:14:55.879] ==================================================
[2026-03-19 14:14:55.879] 


MethodError: MethodError: no method matching besseli0x(::Double64)
The function `besseli0x` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  besseli0x(!Matched::T) where T<:Union{Float32, Float64}
   @ Bessels ~/.julia/packages/Bessels/eaWGd/src/besseli.jl:79


In [15]:
Nstr = join(sort(ns), "_")
run_result_file = joinpath(
    save_path,
    "result_$(result_string)_$(rule)_$(boundary)_N_$(Nstr).jld2"
)

"../samples/jld2/result_lesson6_F0000_gauss_p12_LU_EXEX_N_17_18_19_20_21_22_23.jld2"

## Step 3 :: Inspect raw convergence datapoints

Before performing any extrapolation, it is often useful to examine the
raw numerical behavior.

The following plot shows the integral estimates as a function of the
effective step size. This helps detect anomalies such as:

- non-monotonic convergence,
- resolution ranges dominated by pre-asymptotic effects,
- potential numerical instability.

In [16]:
h_power = 7
xscale = :log
yscale = :linear

plot_datapoints_result(
    run_result;
    name = run_result_file,
    h_power = h_power,
    xscale = xscale,
    yscale = yscale,
    rule = run_result.rule,
    boundary = run_result.boundary,
    figs_dir = save_path,
    save_file = true
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Step 4 :: Generate datapoints-only reports

At this stage we produce documentation artifacts based solely on the
raw data, without any fitted model.

These reports are useful for:

- preliminary validation,
- sharing intermediate results,
- archival of experimental conditions,
- reproducibility before model-dependent analysis.

In [17]:
write_convergence_summary_datapoints(
    run_result;
    name = run_result_file,
    h_power = h_power,
    xscale = xscale,
    yscale = yscale,
    rule = run_result.rule,
    boundary = run_result.boundary,
    format = :md,
    out_dir = save_path,
    save_file = true,
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Generate a datapoints-only internal note

Next we construct a self-contained internal note for the raw dataset
before any fit is applied.

This note collects the datapoints summary together with the saved
datapoints plot inside a buildable LaTeX project.

Such an artifact is useful when you want to archive or share the
pre-fit behavior of a numerical experiment independently of the
final extrapolation step.

In [18]:
note_info = write_convergence_internal_note_datapoints(
    run_result;
    name = run_result_file,
    h_power = h_power,
    xscale = xscale,
    yscale = yscale,
    rule = run_result.rule,
    boundary = run_result.boundary,
    out_dir = save_path,
    save_file = true,
    try_build_pdf = true,
    move_existing_plots = true,
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

### Tag outputs for Lesson 6

To avoid overwriting files generated in previous lessons,
we append a lesson-specific suffix to the result name.

This modified identifier will be used by downstream functions
(such as plotting or report generation) when constructing
output filenames.

In [19]:
result_string = "lesson6_" * result_string

"lesson6_lesson6_F0000"

## Step 5 :: Perform continuum extrapolation

We now estimate the continuum limit \( h \to 0 \) using a weighted
least-\(\chi^2\) fit.

The fitted model uses the error-scale information generated earlier to
construct statistically meaningful weights across resolutions.

Only a subset of the lowest-order terms may be retained to stabilize
the fit.

In [20]:
fit_nterms = 2
fit_nerr_terms = 3

fit_result = least_chi_square_fit(
    run_result; 
    nterms=fit_nterms, 
    ff_shift=0, 
    nerr_terms=fit_nerr_terms
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Step 6 :: Inspect fitted parameters

The printed output summarizes the estimated continuum value and the
coefficients of the asymptotic expansion.

This information can be used to assess:

- the stability of the fit,
- the magnitude of higher-order corrections,
- consistency with theoretical expectations.

In [21]:
print_fit_result(fit_result)

UndefVarError: UndefVarError: `fit_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Step 7 :: Visualize fitted convergence

The convergence plot overlays the fitted model on the raw datapoints
and includes uncertainty bands derived from the covariance matrix.

This visualization provides an intuitive check that the model captures
the observed scaling behavior.

In [22]:
plot_convergence_result(
    run_result, 
    fit_result;
    name=result_string,
    figs_dir=save_path,
    save_file=true
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Step 8 :: Produce fit-aware documentation

Finally, we generate summaries and internal-note artifacts that
incorporate the fitted results.

These outputs are designed to be directly usable in technical reports
or publications, ensuring that the numerical study is fully
reproducible.

In [23]:
write_convergence_summary(
    run_result,
    fit_result;
    name = result_string,
    format = :md,
    out_dir = save_path,
    save_file = true,
    nterms=fit_nterms, 
    nerr_terms=fit_nerr_terms,
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Build the final internal note

Finally, we generate a complete internal note for the fitted analysis.

This note combines

- the fit-aware summary tables,
- the saved convergence figures,
- a REVTeX-based master document,
- and a reproducible build structure.

At this point the workflow has produced both

- pre-fit reporting artifacts, and
- final fit-based reporting artifacts

from a single numerical experiment.

In [24]:
note_info = write_convergence_internal_note(
    run_result,
    fit_result;
    name = result_string,
    rule = run_result.rule,
    boundary = run_result.boundary,
    out_dir = save_path,
    save_file = true,
    try_build_pdf = true,
    move_existing_plots = true,
    nterms=fit_nterms, 
    nerr_terms=fit_nerr_terms,
    # author = "Benjamin J. Choi",
    # affiliation = "Center for Computational Sciences, University of Tsukuba, 1-1-1 Tennodai, Tsukuba, Ibaraki 305-8577, Japan",
    # abstract_text = "This note summarizes the convergence study and fitted extrapolation result."
)

UndefVarError: UndefVarError: `run_result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

## Summary

In this notebook we demonstrated a full research-style reporting
workflow using `Maranatha.jl`.

Starting from a realistic quadrature setup for a challenging integrand,
we performed:

1. definition and execution of a multi-resolution experiment,
2. inspection of raw convergence datapoints,
3. generation of datapoints-only documentation,
4. continuum extrapolation via least-\(\chi^2\) fitting,
5. visualization of fitted convergence behavior,
6. production of fit-aware summaries and internal notes.

This workflow illustrates how a single numerical experiment can be
developed into a complete, reproducible research package covering both

- the pre-fit validation stage, and
- the final model-based analysis stage.

Such a pipeline is particularly valuable when results must be archived,
shared with collaborators, or incorporated into formal reports.